In [36]:
import os
import time
import copy
import random
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

In [37]:
%load_ext autoreload
%autoreload 2

In [38]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [39]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [40]:
def find_project_root():
    current = Path.cwd().resolve()

    while current != current.parent:
        if (current / ".git").exists():
            return current

        current = current.parent

    raise FileNotFoundError("Could not find project root.")


PROJECT_ROOT = find_project_root()

DATA_ROOT = PROJECT_ROOT / "data" / "GTSRB"
SPLIT_DIR = PROJECT_ROOT / "splits"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Project root:", PROJECT_ROOT)

Project root: D:\ML\PR\CNN_Project


In [41]:
if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT)
    )

In [42]:
from Common.dataset import create_dataloaders

from Common.models import (
    ModelB,
    count_parameters
)

from Common.training import train_model

In [43]:
train_df = pd.read_csv(
    SPLIT_DIR / "train_split.csv"
)

val_df = pd.read_csv(
    SPLIT_DIR / "val_split.csv"
)

test_df = pd.read_csv(
    SPLIT_DIR / "test_split.csv"
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 27446
Validation: 5881
Test: 5882


In [44]:
train_loader, val_loader, test_loader = create_dataloaders(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    data_root=DATA_ROOT,
    batch_size=32,
    num_workers=0
)

In [45]:
# # Training transformations
# train_transform = transforms.Compose([
#     transforms.Resize((64, 64)),
#
#     transforms.ColorJitter(
#         brightness=0.2,
#         contrast=0.2
#     ),
#
#     transforms.RandomRotation(
#         degrees=8
#     ),
#
#     transforms.RandomAffine(
#         degrees=0,
#         translate=(0.05, 0.05),
#         scale=(0.95, 1.05)
#     ),
#
#     transforms.ToTensor(),
#
#     transforms.Normalize(
#         mean=[0.5, 0.5, 0.5],
#         std=[0.5, 0.5, 0.5]
#     )
# ])
#
#
# # Validation and test transformations
# eval_transform = transforms.Compose([
#     transforms.Resize((64, 64)),
#
#     transforms.ToTensor(),
#
#     transforms.Normalize(
#         mean=[0.5, 0.5, 0.5],
#         std=[0.5, 0.5, 0.5]
#     )
# ])

In [46]:
# class GTSRBDataset(Dataset):
#
#     def __init__(
#         self,
#         dataframe,
#         data_root,
#         transform=None
#     ):
#         self.dataframe = dataframe.reset_index(drop=True)
#         self.data_root = Path(data_root)
#         self.transform = transform
#
#     def __len__(self):
#         return len(self.dataframe)
#
#     def __getitem__(self, index):
#
#         row = self.dataframe.iloc[index]
#
#         # Construct complete image path
#         image_path = (
#             self.data_root
#             / row["Path"]
#         )
#
#         # Read image and force RGB format
#         image = Image.open(
#             image_path
#         ).convert("RGB")
#
#         # Class ID
#         label = int(
#             row["ClassId"]
#         )
#
#         # Apply preprocessing / augmentation
#         if self.transform is not None:
#             image = self.transform(image)
#
#         return image, label

In [47]:
# train_dataset = GTSRBDataset(
#     dataframe=train_df,
#     data_root=DATA_ROOT,
#     transform=train_transform
# )
#
# val_dataset = GTSRBDataset(
#     dataframe=val_df,
#     data_root=DATA_ROOT,
#     transform=eval_transform
# )
#
# test_dataset = GTSRBDataset(
#     dataframe=test_df,
#     data_root=DATA_ROOT,
#     transform=eval_transform
# )

In [48]:
# BATCH_SIZE = 32
# NUM_WORKERS = 0
#
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     num_workers=NUM_WORKERS,
#     pin_memory=torch.cuda.is_available()
# )
#
# val_loader = DataLoader(
#     val_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=NUM_WORKERS,
#     pin_memory=torch.cuda.is_available()
# )
#
# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=NUM_WORKERS,
#     pin_memory=torch.cuda.is_available()
# )

In [49]:
# images, labels = next(iter(train_loader))
#
# print("Image batch shape:", images.shape)
# print("Label batch shape:", labels.shape)
#
# print("Image data type:", images.dtype)
# print("Label data type:", labels.dtype)
#
# print("First 10 labels:", labels[:10])

Depthwise separable CNN

In [50]:
# class DepthwiseSeparableConv(
#     nn.Module
# ):
#
#     def __init__(
#         self,
#         in_channels,
#         out_channels
#     ):
#
#         super().__init__()
#
#         self.depthwise = nn.Conv2d(
#             in_channels,
#             in_channels,
#             kernel_size=3,
#             padding=1,
#             groups=in_channels,
#             bias=False
#         )
#
#         self.pointwise = nn.Conv2d(
#             in_channels,
#             out_channels,
#             kernel_size=1
#         )
#
#         self.relu = nn.ReLU()
#
#     def forward(
#         self,
#         x
#     ):
#
#         x = self.depthwise(x)
#
#         x = self.pointwise(x)
#
#         x = self.relu(x)
#
#         return x

In [51]:
model_b = ModelB().to(device)

print(model_b)

parameters_b = count_parameters(
    model_b
)

print(
    "Model B trainable parameters:",
    parameters_b
)

ModelB(
  (features): Sequential(
    (0): DepthwiseSeparableConv(
      (depthwise): Conv2d(3, 3, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=3, bias=False)
      (pointwise): Conv2d(3, 32, kernel_size=(1, 1), stride=(1, 1))
      (relu): ReLU()
    )
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): DepthwiseSeparableConv(
      (depthwise): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
      (pointwise): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1))
      (relu): ReLU()
    )
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): DepthwiseSeparableConv(
      (depthwise): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
      (pointwise): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1))
      (relu): ReLU()
    )
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6

In [52]:
# class ModelB(nn.Module):
#
#     def __init__(
#         self,
#         num_classes=43
#     ):
#
#         super().__init__()
#
#         self.features = nn.Sequential(
#
#             DepthwiseSeparableConv(
#                 3,
#                 32
#             ),
#
#             nn.MaxPool2d(2),
#
#             DepthwiseSeparableConv(
#                 32,
#                 64
#             ),
#
#             nn.MaxPool2d(2),
#
#             DepthwiseSeparableConv(
#                 64,
#                 128
#             ),
#
#             nn.MaxPool2d(2),
#
#             DepthwiseSeparableConv(
#                 128,
#                 160
#             ),
#
#             nn.MaxPool2d(2)
#         )
#
#         self.pool = (
#             nn.AdaptiveAvgPool2d(
#                 (1, 1)
#             )
#         )
#
#         self.classifier = nn.Sequential(
#
#             nn.Flatten(),
#
#             nn.Linear(
#                 160,
#                 128
#             ),
#
#             nn.ReLU(),
#
#             nn.Dropout(0.3),
#
#             nn.Linear(
#                 128,
#                 num_classes
#             )
#         )
#
#     def forward(
#         self,
#         x
#     ):
#
#         x = self.features(x)
#
#         x = self.pool(x)
#
#         x = self.classifier(x)
#
#         return x

In [53]:
# def count_parameters(model):
#     return sum(
#         parameter.numel()
#         for parameter in model.parameters()
#         if parameter.requires_grad
#     )
#
# model_b = ModelB().to(
#     device
# )
#
# print(
#     count_parameters(
#         model_b
#     )
# )

In [54]:
# criterion = nn.CrossEntropyLoss()

### Training func

In [55]:
# def train_one_epoch(
#     model,
#     loader,
#     criterion,
#     optimizer,
#     device
# ):
#     model.train()
#
#     running_loss = 0.0
#     correct = 0
#     total = 0
#
#     for images, labels in loader:
#
#         images = images.to(device)
#         labels = labels.to(device)
#
#         optimizer.zero_grad(set_to_none=True)
#
#         outputs = model(images)
#
#         loss = criterion(
#             outputs,
#             labels
#         )
#
#         loss.backward()
#
#         optimizer.step()
#
#         running_loss += (
#             loss.item()
#             * images.size(0)
#         )
#
#         predictions = outputs.argmax(
#             dim=1
#         )
#
#         correct += (
#             predictions == labels
#         ).sum().item()
#
#         total += labels.size(0)
#
#     epoch_loss = (
#         running_loss / total
#     )
#
#     epoch_accuracy = (
#         correct / total
#     )
#
#     return (
#         epoch_loss,
#         epoch_accuracy
#     )

### Validation func

In [56]:
# def evaluate_epoch(
#     model,
#     loader,
#     criterion,
#     device
# ):
#
#     model.eval()
#
#     running_loss = 0.0
#     correct = 0
#     total = 0
#
#     with torch.no_grad():
#
#         for images, labels in loader:
#
#             images = images.to(
#                 device
#             )
#
#             labels = labels.to(
#                 device
#             )
#
#             outputs = model(
#                 images
#             )
#
#             loss = criterion(
#                 outputs,
#                 labels
#             )
#
#             running_loss += (
#                 loss.item()
#                 * images.size(0)
#             )
#
#             predictions = (
#                 outputs.argmax(
#                     dim=1
#                 )
#             )
#
#             correct += (
#                 predictions
#                 == labels
#             ).sum().item()
#
#             total += labels.size(0)
#
#     loss = (
#         running_loss
#         / total
#     )
#
#     accuracy = (
#         correct
#         / total
#     )
#
#     return (
#         loss,
#         accuracy
#     )

In [57]:
# def train_model(
#     model,
#     train_loader,
#     val_loader,
#     optimizer,
#     criterion,
#     epochs,
#     device
# ):
#
#     history = {
#         "train_loss": [],
#         "train_accuracy": [],
#         "val_loss": [],
#         "val_accuracy": [],
#         "epoch_time": []
#     }
#
#     best_val_loss = float(
#         "inf"
#     )
#
#     best_weights = copy.deepcopy(
#         model.state_dict()
#     )
#
#     for epoch in range(
#         epochs
#     ):
#
#         start_time = (
#             time.perf_counter()
#         )
#
#         train_loss, train_acc = (
#             train_one_epoch(
#                 model,
#                 train_loader,
#                 criterion,
#                 optimizer,
#                 device
#             )
#         )
#
#         val_loss, val_acc = (
#             evaluate_epoch(
#                 model,
#                 val_loader,
#                 criterion,
#                 device
#             )
#         )
#
#         elapsed = (
#             time.perf_counter()
#             - start_time
#         )
#
#         history[
#             "train_loss"
#         ].append(train_loss)
#
#         history[
#             "train_accuracy"
#         ].append(train_acc)
#
#         history[
#             "val_loss"
#         ].append(val_loss)
#
#         history[
#             "val_accuracy"
#         ].append(val_acc)
#
#         history[
#             "epoch_time"
#         ].append(elapsed)
#
#         if val_loss < best_val_loss:
#
#             best_val_loss = (
#                 val_loss
#             )
#
#             best_weights = copy.deepcopy(
#                 model.state_dict()
#             )
#
#         print(
#             f"Epoch {epoch + 1:02d}/{epochs} | "
#             f"Train Loss: {train_loss:.4f} | "
#             f"Train Acc: {train_acc:.4f} | "
#             f"Val Loss: {val_loss:.4f} | "
#             f"Val Acc: {val_acc:.4f} | "
#             f"Time: {elapsed:.2f}s"
#         )
#
#     model.load_state_dict(
#         best_weights
#     )
#
#     return model, history

2 - epoch smoke test

In [58]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model_b.parameters(),
    lr=1e-3
)

model_b, test_history = train_model(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=2,
    device=device
)

Epoch 01/2 | Train Loss: 3.3120 | Train Acc: 0.0973 | Val Loss: 2.9928 | Val Acc: 0.1493 | Time: 55.17s
Epoch 02/2 | Train Loss: 2.6445 | Train Acc: 0.2146 | Val Loss: 2.0573 | Val Acc: 0.3567 | Time: 59.24s


In [ ]:
from Common.training import (
    train_model,
    create_optimizer
)

from Common.evaluation import (
    get_predictions,
    calculate_metrics,
    plot_training_curves,
    plot_confusion_matrix,
    print_classification_report,
    total_parameters,
    estimated_fp32_size_kb,
    save_model_and_get_size,
    measure_inference_time
)

In [ ]:
selected_optimizer = (
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "selected_optimizer.txt"
).read_text().strip()

print(
    "Final optimizer:",
    selected_optimizer
)

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


model_b = ModelB().to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer_b = create_optimizer(
    selected_optimizer,
    model_b.parameters()
)

In [ ]:
model_b, history_b = train_model(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_b,
    criterion=criterion,
    epochs=30,
    device=device
)

In [ ]:
FIGURES_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
)

plot_training_curves(
    history=history_b,
    model_name="Model B",
    save_dir=FIGURES_DIR
)

In [ ]:
y_true_b, y_pred_b = get_predictions(
    model_b,
    test_loader,
    device
)

metrics_b = calculate_metrics(
    y_true_b,
    y_pred_b
)

print(metrics_b)

In [ ]:
plot_confusion_matrix(
    y_true_b,
    y_pred_b,
    model_name="Model B",
    save_dir=FIGURES_DIR
)

In [ ]:
print_classification_report(
    y_true_b,
    y_pred_b
)

In [ ]:
average_epoch_time_b = np.mean(
    history_b["epoch_time"]
)

print(
    "Average epoch time:",
    average_epoch_time_b
)

In [ ]:
parameters_b = total_parameters(
    model_b
)

estimated_size_b = (
    estimated_fp32_size_kb(
        model_b
    )
)

model_b_path = (
    PROJECT_ROOT
    / "checkpoints"
    / "model_b_final.pth"
)

actual_size_b = (
    save_model_and_get_size(
        model_b,
        model_b_path
    )
)

print(
    "Parameters:",
    parameters_b
)

print(
    "Estimated size:",
    estimated_size_b,
    "KB"
)

print(
    "Actual size:",
    actual_size_b,
    "KB"
)

In [ ]:
inference_time_b = (
    measure_inference_time(
        model_b,
        test_loader,
        device
    )
)

print(
    "Inference time:",
    inference_time_b,
    "ms/image"
)

In [ ]:
model_b_results = {
    "Model": "Model B",
    "Parameters": parameters_b,
    "Estimated FP32 Size (KB)": estimated_size_b,
    "Actual Model Size (KB)": actual_size_b,
    "Average Epoch Time (s)": average_epoch_time_b,
    "Inference Time (ms/image)": inference_time_b,
    "Test Accuracy": metrics_b["Test Accuracy"],
    "Macro Precision": metrics_b["Macro Precision"],
    "Macro Recall": metrics_b["Macro Recall"]
}

model_b_df = pd.DataFrame(
    [model_b_results]
)

model_b_df.to_csv(
    PROJECT_ROOT
    / "results"
    / "metrics"
    / "model_b_metrics.csv",
    index=False
)

model_b_df